# Phase 08 — Train / Val / Test Split & Dataset Versioning
## CodeMentor-LLM
Splitting cleaned dataset into train, validation and test sets.

## Split Strategy
- Train : 80%
- Val   : 10%
- Test  : 10%
- Seed  : 42 (fixed for reproducibility)

# Libraries

In [ ]:
!pip install -q datasets==3.3.2 pandas==2.2.3

#  Login and load dataset

In [ ]:
from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset

# Login
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in successfully")

# Load cleaned dataset
print("\nLoading cleaned dataset...")
dataset = load_dataset("Abdulmoiz123/codementor-llm-cleaned")
print(f"Dataset loaded successfully")
print(f"Total samples: {len(dataset['train'])}")

# Split dataset

In [ ]:
from datasets import DatasetDict

# Split dataset
# First split — 80% train, 20% temp
train_test = dataset["train"].train_test_split(test_size=0.2, seed=42)

# Second split — split 20% temp into 10% val and 10% test
val_test = train_test["test"].train_test_split(test_size=0.5, seed=42)

# Create final dataset dict
final_dataset = DatasetDict({
    "train": train_test["train"],
    "validation": val_test["train"],
    "test": val_test["test"]
})

print("Split Summary:")
print(f"  Train      : {len(final_dataset['train'])} samples")
print(f"  Validation : {len(final_dataset['validation'])} samples")
print(f"  Test       : {len(final_dataset['test'])} samples")
print(f"  Total      : {len(final_dataset['train']) + len(final_dataset['validation']) + len(final_dataset['test'])} samples")

# Push splits to HuggingFace Hub

In [ ]:
# Push to HF Hub
print("Pushing splits to HuggingFace Hub...")
final_dataset.push_to_hub("Abdulmoiz123/codementor-llm-splits")
print("Dataset splits pushed successfully")
print(f"\nDataset available at:")
print(f"https://huggingface.co/datasets/Abdulmoiz123/codementor-llm-splits")

#  Save splits locally

In [ ]:
import json

# Save train split
with open("train.jsonl", "w") as f:
    for sample in final_dataset["train"]:
        f.write(json.dumps(sample) + "\n")
print(f"Train split saved — {len(final_dataset['train'])} samples")

# Save validation split
with open("validation.jsonl", "w") as f:
    for sample in final_dataset["validation"]:
        f.write(json.dumps(sample) + "\n")
print(f"Validation split saved — {len(final_dataset['validation'])} samples")

# Save test split
with open("test.jsonl", "w") as f:
    for sample in final_dataset["test"]:
        f.write(json.dumps(sample) + "\n")
print(f"Test split saved — {len(final_dataset['test'])} samples")